In [1]:
import pandas as pd

In [2]:
# Cargamos los datos

data = pd.read_csv("processed_dataset.csv", sep=",", engine="python")

In [3]:
data.head()

,created_date,created_timestamp,subreddit,title,author,author_created_utc,full_link,score,num_comments,num_crossposts,subreddit_subscribers,post,sentiment,clean_post
0,2010-02-11 19:47:22,1265910442.0,analytics,So what do you guys all do related to analytic...,xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b0...,7.0,4.0,0.0,NaN,There's a lot of reasons to want to know all t...,NEGATIVE,there lot reason want know stuff figured id ge...
1,2010-03-04 20:17:26,1267726646.0,analytics,"Google's Invasive, non-Anonymized Ad Targeting...",xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b9...,2.0,1.0,0.0,NaN,"I'm cross posting this from /r/cyberlaw, hopef...",NEGATIVE,im cross posting rcyberlaw hopefully guy find ...
2,2011-01-06 04:51:18,1294282278.0,analytics,"DotCed - Functional Web Analytics - Tagging, R...",dotced,1.294282e+09,https://www.reddit.com/r/analytics/comments/ew...,1.0,1.0,NaN,NaN,"DotCed,a Functional Analytics Consultant, offe...",NEGATIVE,dotceda functional analytics consultant offeri...
3,2011-01-19 11:45:30,1295430330.0,analytics,Program Details - Data Analytics Course,iqrconsulting,1.288245e+09,https://www.reddit.com/r/analytics/comments/f5...,0.0,0.0,NaN,NaN,Here is the program details of the data analyt...,NEGATIVE,program detail data analytics certification co...
4,2011-01-19 21:52:28,1295466748.0,analytics,potential job in web analytics... need to anal...,therewontberiots,1.278672e+09,https://www.reddit.com/r/analytics/comments/f5...,2.0,4.0,NaN,NaN,i decided grad school (physics) was not for me...,POSITIVE,decided grad school physic branching job marke...


In [4]:
text1 = data["clean_post"][4]
text1
text2 = data["clean_post"][2]
text2

'dotceda functional analytics consultant offering google analytics tagging reporting analysis strategy seo auditing sem optimization call min consultation'

Distancia Semántica Usando TF-IDF y Similitud del Coseno

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

In [6]:
def texts_distance(text1: str, text2: str) -> float:
    # Crear un vectorizador TF-IDF
    vectorizer = TfidfVectorizer(stop_words="english")   # Excluimos stopwords en inglés
    
    # Convertir los textos en vectores TF-IDF
    tfidf_matrix = vectorizer.fit_transform([text1, text2])
    
    # Calcular la similitud del coseno entre los dos vectores
    cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
    
    distance = 1 - cosine_sim[0][0]
    
    return distance


# Calcular la distancia semántica
distance = texts_distance(text1, text2)
print(f"La distancia semántica entre los dos textos es: {distance}")

La distancia semántica entre los dos textos es: 0.9415203300295236


In [7]:
def texts_distance_euc(text1: str, text2: str) -> float:
    # Crear un vectorizador TF-IDF
    vectorizer = TfidfVectorizer(stop_words="english")   # Excluimos stopwords en inglés
    
    # Convertir los textos en vectores TF-IDF
    tfidf_matrix = vectorizer.fit_transform([text1, text2])
    
    # Calcular la similitud del coseno entre los dos vectores
    cosine_sim = euclidean_distances(tfidf_matrix[0:1], tfidf_matrix[1:2])
    
    distance = cosine_sim[0][0]
    
    return distance


# Calcular la distancia semántica
distance = texts_distance_euc(text1, text2)
print(f"La distancia semántica entre los dos textos es: {distance}")

La distancia semántica entre los dos textos es: 1.372239286735024


In [8]:
def hamming_distance(str1, str2):
    # Verificar si las cadenas tienen la misma longitud
    if len(str1) != len(str2):
        raise ValueError("Las cadenas deben tener la misma longitud")
    
    # Contar las diferencias
    return sum(el1 != el2 for el1, el2 in zip(str1, str2))


# Ejemplo
str1 = "orquestas"
str2 = "ordenador"
distance = hamming_distance(str1, str2)
print(f"Distancia de Hamming: {distance}")

Distancia de Hamming: 7


In [9]:
def levenshtein_distance(str1, str2):
    # Crear una matriz de tamaño (len(str1)+1) x (len(str2)+1)
    m, n = len(str1), len(str2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    # Llenar la primera fila y columna
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    
    # Llenar la matriz con las operaciones mínimas
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if str1[i - 1] == str2[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1,   # Eliminación
                           dp[i][j - 1] + 1,   # Inserción
                           dp[i - 1][j - 1] + cost)   # Sustitución
    
    return dp[m][n]


# Ejemplo
str1 = "orquestas"
str2 = "ordenador"
distance = levenshtein_distance(str1, str2)
print(f"Distancia de Levenshtein: {distance}")

Distancia de Levenshtein: 7


In [10]:
import gensim.downloader as api
from gensim.models.keyedvectors import KeyedVectors
from scipy.spatial.distance import cdist

# Cargar el modelo preentrenado Word2Vec de Gensim (en inglés)
model = api.load("word2vec-google-news-300")

def word_movers_distance(text1, text2, model):
    # Tokenizar los textos y obtener sus vectores promedio
    words1 = text1.lower().split()
    words2 = text2.lower().split()
    
    # Filtrar palabras que no están en el vocabulario del modelo
    words1 = [word for word in words1 if word in model.key_to_index]
    words2 = [word for word in words2 if word in model.key_to_index]
    
    if not words1 or not words2:
        return float("inf")   # Si algún texto no tiene palabras válidas en el vocabulario
    
    # Obtener las representaciones de las palabras
    vectors1 = [model[word] for word in words1]
    vectors2 = [model[word] for word in words2]
    
    # Calcular la distancia entre los puntos
    distance_matrix = cdist(vectors1, vectors2, metric="euclidean")
    
    # El coste mínimo de mover las palabras
    return distance_matrix.min()


# Ejemplo
text1 = "I love programming"
text2 = "I enjoy coding"
distance = word_movers_distance(text1, text2, model)
print(f"Word Mover's Distance: {distance}")

Word Mover's Distance: 0.0


Distancia Semántica Usando Word2Vec y Similitud del Coseno

In [12]:
from gensim.models import Word2Vec
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def get_average_word_vector(text, model):
    words = text.lower().split()   # Convertir el texto a minúsculas y dividirlo en palabras
    word_vectors = []   # Lista para almacenar los vectores de palabras
    
    for word in words:
        if word in model.wv:   # Verificar si la palabra está en el vocabulario del modelo
            word_vectors.append(model.wv[word])   # Agregar el vector de la palabra a la lista
    
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)   # Si no hay palabras en el vocabulario, devuelve un vector cero
    
    return np.mean(word_vectors, axis=0)   # Calcular y devolver el vector promedio de las palabras


def texts_distance_word2vec(text1: str, text2: str, model: Word2Vec) -> float:
    '''Función para obtener el vector promedio de las palabras en el texto'''
    
    # Obtener los vectores promedio de cada texto
    vector1 = get_average_word_vector(text1, model)
    vector2 = get_average_word_vector(text2, model)
    
    # Calcular la similitud del coseno entre los vectores
    cosine_sim = cosine_similarity([vector1], [vector2])
    
    distance = 1 - cosine_sim[0][0]
    
    return distance


# Para el ejemplo, crearemos un modelo Word2Vec básico
sentences = [
    ["the", "quick", "brown", "fox"],
    ["jumps", "over", "the", "lazy", "dog"],
    ["the", "swift", "auburn", "fox"],
    ["leaps", "over", "a", "lazy", "canine"]
]

# Calcular la distancia semántica utilizando Word2Vec
model = Word2Vec(sentences, min_count=1)
distance_word2vec = texts_distance_word2vec(text1, text2, model)
print(f"Distancia semántica con Word2Vec: {distance_word2vec}")

Distancia semántica con Word2Vec: 1.0


Distancia Semántica Usando Doc2Vec y Similitud del Coseno

In [13]:
from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity

# Convertir las oraciones en documentos etiquetados
tagged_data = [TaggedDocument(words=sentence, tags=[str(i)]) for i, sentence in enumerate(sentences)]

# Entrenar el modelo Doc2Vec
model = Doc2Vec(tagged_data, vector_size=20, min_count=1, epochs=100)

def texts_distance_doc2vec(text1: str, text2: str, model: Doc2Vec) -> float:
    # Obtener los vectores de documento
    vector1 = model.infer_vector(text1.split())
    vector2 = model.infer_vector(text2.split())
    
    # Calcular la similitud del coseno entre los vectores de los documentos
    cosine_sim = cosine_similarity([vector1], [vector2])
  
    distance = 1 - cosine_sim[0][0]
    
    return distance

distance_doc2vec = texts_distance_doc2vec(text1, text2, model)
print(f"Distancia semántica con Doc2Vec: {distance_doc2vec}")

Distancia semántica con Doc2Vec: 1.248281016945839
